In [3]:
# Install Dependencies
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [4]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [28]:
# Create API Client
from anthropic import Anthropic

client = Anthropic()
model = "anthropic.claude-4-5-sonnet"

In [29]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, stop_sequences=None):
    params = {
        "model":model,
        "max_tokens": 1000,
        "messages": messages
    }
    if system:
        params["system"] = system
    
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    
    return message.content[0].text


In [30]:
# make the initial list of messages
messages = []

# add the initial user question
add_user_message(messages, "Create a json config for AWS EventBridge")
add_assistant_message(messages, "```json")

# pass the messages to chat
answer = chat(messages, stop_sequences=["```"])

answer



'\n{\n  "EventBridgeConfig": {\n    "EventBus": {\n      "Name": "custom-event-bus",\n      "Description": "Custom event bus for application events"\n    },\n    "Rules": [\n      {\n        "Name": "process-orders-rule",\n        "Description": "Rule to process order events",\n        "State": "ENABLED",\n        "EventPattern": {\n          "source": ["custom.orders"],\n          "detail-type": ["Order Placed", "Order Updated"],\n          "detail": {\n            "status": ["pending", "processing"]\n          }\n        },\n        "Targets": [\n          {\n            "Id": "1",\n            "Arn": "arn:aws:lambda:us-east-1:123456789012:function:ProcessOrderFunction",\n            "RetryPolicy": {\n              "MaximumRetryAttempts": 2,\n              "MaximumEventAge": 3600\n            },\n            "DeadLetterConfig": {\n              "Arn": "arn:aws:sqs:us-east-1:123456789012:order-dlq"\n            }\n          }\n        ]\n      },\n      {\n        "Name": "scheduled-b

In [31]:
import json

json.loads(answer.strip())

{'EventBridgeConfig': {'EventBus': {'Name': 'custom-event-bus',
   'Description': 'Custom event bus for application events'},
  'Rules': [{'Name': 'process-orders-rule',
    'Description': 'Rule to process order events',
    'State': 'ENABLED',
    'EventPattern': {'source': ['custom.orders'],
     'detail-type': ['Order Placed', 'Order Updated'],
     'detail': {'status': ['pending', 'processing']}},
    'Targets': [{'Id': '1',
      'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:ProcessOrderFunction',
      'RetryPolicy': {'MaximumRetryAttempts': 2, 'MaximumEventAge': 3600},
      'DeadLetterConfig': {'Arn': 'arn:aws:sqs:us-east-1:123456789012:order-dlq'}}]},
   {'Name': 'scheduled-backup-rule',
    'Description': 'Daily backup task',
    'State': 'ENABLED',
    'ScheduleExpression': 'cron(0 2 * * ? *)',
    'Targets': [{'Id': '1',
      'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:BackupFunction',
      'Input': '{"backup_type": "daily"}'}]},
   {'Name': 'notificat